# Null and synthetic controls (Steps 43-48, 57)

Trailing-window features move smoothly, so an HMM finds 'persistent states' in almost anything. Persistence is read against row-shuffled and row-block-bootstrapped features and against pipeline nulls (random walk, shuffled and block-bootstrapped returns through the same engines); synthetic series with known answers check the machinery itself.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.regimes` and `xauusd_quant.research.regime_*` and read back from `results/regime_research/`. Generate those first:

```
xq regime-research --timeframe 1h --model hmm --states 3
xq regime-research --timeframe 15m --model gmm --components 4
xq regime-walk-forward --timeframe 5m --model hmm --states 4
xq regime-research --all --resume        # the full grid (hours; see --dry-run)
xq regime-research --controls            # synthetic controls
xq regime-research --compare             # cross-timeframe tables
```

**Causal vs offline.** Walk-forward results use models fitted on earlier data only, a frozen scaler and the HMM *forward filter* P(S_t | X_<=t): they are causal and the only thing stored under `data/features/regime/`. Full-sample fits, smoothed probabilities and Viterbi paths are **OFFLINE / NON-CAUSAL RESEARCH ONLY** and never features. States are numbered (`state_0`, `state_1`, ...), never named; their descriptions come from the profiles. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.regimes import load_regime_config

regime = load_regime_config(ROOT / "config" / "regimes.yaml")

TIMEFRAME = "1h"       # 5m | 15m | 30m | 1h - change and re-run
MODEL = "hmm"          # kmeans | gmm | gmm_diag | hmm
K = 3                  # number of states

study = regime.results_dir(TIMEFRAME, MODEL, K)
tf_dir = regime.timeframe_dir(TIMEFRAME)
if not study.exists():
    print(f"No results for {TIMEFRAME} {MODEL} K={K}. Run:  xq regime-research "
          f"--timeframe {TIMEFRAME} --model {MODEL} --states {K}")

def table(name, folder=None):
    """Load a previously generated regime table (None if it was not written)."""
    path = (folder or study) / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name, folder=None):
    path = (folder or study / "plots") / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

summary_path = study / "summary.json"
summary = json.loads(summary_path.read_text())["summary"] if summary_path.exists() else {}
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## HMM persistence: real vs nulls

In [ ]:
nulls = table('null_controls', tf_dir / 'nulls')
nulls.select('source', 'states', 'median_expected_duration', 'median_observed_duration', 'persistence_excess',
             'temporal_gain_per_obs', 'min_bhattacharyya', 'adjacency_share', 'flags').sort('states', 'source') if nulls is not None else None

## Synthetic controls

In [ ]:
syn = regime.results_path / 'synthetic_controls'
for name in ('selection', 'persistence', 'recovery'):
    path = syn / f'{name}.csv'
    if path.exists():
        display(pl.read_csv(path))

## Every registered hypothesis and its verdict

In [ ]:
table('hypothesis_verdicts').select('hypothesis_id', 'feature', 'target', 'horizon', 'metric', 'value', 'control_value', 'verdict')

In [ ]:
path = regime.results_path / 'hypothesis_test_counts.csv'
pl.read_csv(path) if path.exists() else None